1\. Write a function that converts number representation, bin<->dec<->hex. (Clearly using the corresponding python built-in functions is not fair..)

In [4]:
# First let's define representation of other bases and their prefix
# Maybe would have been better to have a single dictionary basencode also with representation...
hexstd = {"A": 10, "B": 11, "C": 12, "D": 13, "E": 14, "F": 15}
octstd = {"A": 10, "B": 11, "C": 12, "D": 13, "E": 14, "F": 15, "G":16, "H":17}
others = "0123456789"
basencode = {"0b": 2, "0d": 10, "0x":16, "0c": 18}
basereps = {16: hexstd, 18: octstd}

In [5]:
# To write a general method we need to be able to work with arithmetic in decimal:
# this function returns the number in decimal for any representation loaded above
def ndecimal(nrep):
    base = basencode[nrep[0:2]] # get the base of the input
    if base > 10:
        baserep = basereps[base]

    bits = nrep[2:] # removing prefix
    dec = 0
    i = len(bits) - 1
    for b in bits:
        if b in others:
            dec += int(b)*base**i
        else:
            dec += baserep[b]*base**i
        i = i-1
    return(dec)

# Some example to verify
print("verify conversion to decimal is working:")
print("this should be 25 (from base 2): ", ndecimal("0b11001"))
print("this should be 20 (from base 16): ", ndecimal("0x14"))
print("this should be 17 (from base 16): ", ndecimal("0x11"))
print("also works in base 10 of course: ", ndecimal("0d77"))

verify conversion to decimal is working:
this should be 25 (from base 2):  25
this should be 20 (from base 16):  20
this should be 17 (from base 16):  17
also works in base 10 of course:  77


In [3]:
# Now let's generalize using the ndecimal() funcion for intermediate computation
# nrep is the number representation in a certain base and newbase is the new base integer
def change_baserep(nrep, newbase):
    # load the correct representation for the new base if necessary
    if newbase > 10:
        nbaserep = basereps[newbase]

    n = ndecimal(nrep) # get number in decimal to work with
    bits = ""
    while n > 0:
        rem = n%newbase
        if rem < 10:
            bits = str(rem) + bits
        else:
            rem = [key for key, v in nbaserep.items() if v == rem][0]
            bits = rem + bits
        n = n // newbase

    prefix = [key for key, val in basencode.items() if val == newbase][0] # simple way to get back the key (prefix of base)
    return(prefix + bits)


# Some examples to check functionality
print("decimal representation of 17 from hexa: ", change_baserep("0x11",10))
print("decimal representation of 25 from bin: ", change_baserep("0b10110",10))
print("hexadecimal representation of 45 from decimal: ", change_baserep("0d45",16))
print(change_baserep("0b110110",16))
print(change_baserep("0c45G", 16))
print(change_baserep("0c45G", 2))
print(change_baserep("0xAA", 18))
print(change_baserep("0xAA", 10))

decimal representation of 17 from hexa:  0d17
decimal representation of 25 from bin:  0d22
hexadecimal representation of 45 from decimal:  0x2D
0x36
0x57A
0b10101111010
0c98
0d170


2\. Write a function that converts a 32 bit word into a single precision floating point (i.e. interprets the various bits as sign, mantissa and exponent)

In [11]:
# function takes word in as sequence of bits (floating point) and computes the corresponding number
# [ ] make it a lil better and with safety features
def floating(floatrep):
    s = floatrep[1]
    expb = floatrep[2:9]
    Mb = floatrep[9:]
    exp = ndecimal("0b" + expb)
    M = i = 0
    while i < len(Mb):
        M += int(Mb[i])*2**(-i-1)
        i += 1
    num = ((-1)**int(s))*(1+M)*(2**(exp-127))
    return num

# try it out: this should yield
word = "10111111101000000000000000000000"
print(len(word))
print(floating(word))

32
1.25


3\. Write a program to determine the underflow and overflow limits (within a factor of 2) for python on your computer. 

**Tips**: define two variables inizialized to 1 and halve/double them enough time to exceed the under/over-flow limits  

In [10]:
# first of all: what happens to float/integer variables exceeding the limit?
a = 2
for i in range(200):
    print(a**i)


1
2
4
8
16
32
64
128
256
512
1024
2048
4096
8192
16384
32768
65536
131072
262144
524288
1048576
2097152
4194304
8388608
16777216
33554432
67108864
134217728
268435456
536870912
1073741824
2147483648
4294967296
8589934592
17179869184
34359738368
68719476736
137438953472
274877906944
549755813888
1099511627776
2199023255552
4398046511104
8796093022208
17592186044416
35184372088832
70368744177664
140737488355328
281474976710656
562949953421312
1125899906842624
2251799813685248
4503599627370496
9007199254740992
18014398509481984
36028797018963968
72057594037927936
144115188075855872
288230376151711744
576460752303423488
1152921504606846976
2305843009213693952
4611686018427387904
9223372036854775808
18446744073709551616
36893488147419103232
73786976294838206464
147573952589676412928
295147905179352825856
590295810358705651712
1180591620717411303424
2361183241434822606848
4722366482869645213696
9444732965739290427392
18889465931478580854784
37778931862957161709568
75557863725914323419136
151

4\. Write a program to determine the machine precision

**Tips**: define a new variable by adding a smaller and smaller value (proceeding similarly to prob. 2) to an original variable and check the point where the two are the same 

5\. Write a function that takes in input three parameters $a$, $b$ and $c$ and prints out the two solutions to the quadratic equation $ax^2+bx+c=0$ using the standard formula:
$$
x=\frac{-b\pm\sqrt{b^2-4ac}}{2a}
$$

(a) use the program to compute the solution for $a=0.001$, $b=1000$ and $c=0.001$

(b) re-express the standard solution formula by multiplying top and bottom by $-b\mp\sqrt{b^2-4ac}$ and again find the solution for $a=0.001$, $b=1000$ and $c=0.001$. How does it compare with what previously obtained? Why?

(c) write a function that compute the roots of a quadratic equation accurately in all cases

6\. Write a program that implements the function $f(x)=x(x−1)$

(a) Calculate the derivative of the function at the point $x = 1$ using the derivative definition:

$$
\frac{{\rm d}f}{{\rm d}x} = \lim_{\delta\to0} \frac{f(x+\delta)-f(x)}{\delta}
$$

with $\delta = 10^{−2}$. Calculate the true value of the same derivative analytically and compare with the answer your program gives. The two will not agree perfectly. Why not?

(b) Repeat the calculation for $\delta = 10^{−4}, 10^{−6}, 10^{−8}, 10^{−10}, 10^{−12}$ and $10^{−14}$. How does the accuracy scales with $\delta$?

7\. Consider the integral of the semicircle of radius 1:
$$
I=\int_{-1}^{1} \sqrt(1-x^2) {\rm d}x
$$
which it's known to be $I=\frac{\pi}{2}=1.57079632679...$.
Alternatively we can use the Riemann definition of the integral:
$$
I=\lim_{N\to\infty} \sum_{k=1}^{N} h y_k 
$$

with $h=2/N$ the width of each of the $N$ slices the domain is divided into and where
$y_k$ is the value of the function at the $k-$th slice.

(a) Write a programe to compute the integral with $N=100$. How does the result compares to the true value?

(b) How much can $N$ be increased if the computation needs to be run in less than a second? What is the gain in running it for 1 minute? 
